# EdgentRAG embedding API (temporary Colab demo)

Run the cells from top to bottom. This notebook starts the embedding API on port 8001 and creates a temporary HTTPS tunnel so your local machine can test it.

> **Development only.** Colab is an interactive notebook service, not reliable production API hosting. Runtime limits and availability vary, and free managed runtimes may terminate web-service workloads. Cloudflare Quick Tunnels are also temporary and have no uptime guarantee. Use this for short experiments only; deploy the API on a suitable server for a durable service. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html) and [Cloudflare Quick Tunnel limitations](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/#limitations).

This notebook hosts only edgentrag.embedding.app:app. It does not host the main sessions/uploads API or connect the ingestion worker to embeddings yet.

## Before running

1. In Colab's **Secrets** panel (key icon), add EDGENTRAG_EMBEDDING_API_TOKEN with the same long random token that your local machine will use. Enable notebook access to that secret. Do not put the token in a code cell or print it.
2. Have this project in a Git repository that Colab can clone, or upload/unzip it into /content/ai-eng. If cloning, the setup cell asks for the HTTPS clone URL. For a private repository, do not paste a GitHub personal access token into the notebook; upload the project or use a secure GitHub/Colab secret-based method.
3. If available, select a GPU in **Runtime → Change runtime type**. The service falls back to CPU if no CUDA GPU is available.

The project's backend requires Python 3.12 or newer. The setup cell checks the runtime version before installing.

In [ ]:
from pathlib import Path
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required by this project; Colab has {sys.version.split()[0]}. "
        "Select a compatible runtime or use another host."
    )

PROJECT_DIR = Path("/content/ai-eng")
if not (PROJECT_DIR / "app/backend/pyproject.toml").is_file():
    repo_url = input("Project HTTPS Git clone URL: ").strip()
    if not repo_url:
        raise ValueError("A Git clone URL is required when the project is not already uploaded.")
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(PROJECT_DIR)],
        check=True,
    )

assert (PROJECT_DIR / "app/backend/pyproject.toml").is_file(), (
    f"Could not find the backend project under {PROJECT_DIR}. "
    "Upload/extract the repository there or update PROJECT_DIR."
)
print(f"Project found at {PROJECT_DIR}")
print(f"Python {sys.version.split()[0]}")

## Install backend and embedding dependencies

The embedding extra installs Sentence Transformers and its model dependencies in this Colab runtime. The normal local backend install does not include them.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_DIR}/app/backend[embedding]"],
    check=True,
)
print("Embedding package installed.")

## Load the secret and choose the model device

The token is read from Colab Secrets and passed only to the Uvicorn process through its environment. auto selects CUDA when available, otherwise CPU. Model weights load on the first /embed request, not at server startup.

In [ ]:
import os
from google.colab import userdata

token = userdata.get("EDGENTRAG_EMBEDDING_API_TOKEN")
if not token:
    raise RuntimeError(
        "Add EDGENTRAG_EMBEDDING_API_TOKEN in Colab Secrets and grant this notebook access."
    )
os.environ["EDGENTRAG_EMBEDDING_API_TOKEN"] = token
os.environ["EDGENTRAG_EMBEDDING_DEVICE"] = "auto"
os.environ.setdefault(
    "EDGENTRAG_EMBEDDING_MODEL_NAME",
    "sentence-transformers/all-MiniLM-L6-v2",
)
print("Secret loaded (value hidden).")

## Start the FastAPI service

This starts Uvicorn in the background and waits for the local health endpoint. Health does not load model weights. If startup fails, the cell prints the server log.

In [ ]:
import time
import urllib.error
import urllib.request

server_log_path = Path("/tmp/edgentrag-uvicorn.log")
server_log = server_log_path.open("w")
server = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "edgentrag.embedding.app:app",
        "--host", "0.0.0.0",
        "--port", "8001",
    ],
    cwd=PROJECT_DIR / "app/backend",
    env=os.environ.copy(),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

health_url = "http://127.0.0.1:8001/health"
for _ in range(60):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            print("Embedding API is up:", response.read().decode())
            break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    server.terminate()
    raise TimeoutError("Embedding API did not become healthy within 60 seconds.")

if server.poll() is not None:
    server_log.flush()
    print(server_log_path.read_text())
    raise RuntimeError("Uvicorn exited during startup.")

## Create the temporary HTTPS tunnel

This tunnel forwards requests to Uvicorn on Colab's localhost:8001. The output URL is public, so keep the bearer token private. The /embed route checks the token; /health is intentionally unauthenticated. The URL changes when the tunnel/runtime restarts.

In [ ]:
import re
import urllib.request

cloudflared_deb = Path("/tmp/cloudflared-linux-amd64.deb")
cloudflared_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64.deb"
)
urllib.request.urlretrieve(cloudflared_url, cloudflared_deb)
subprocess.run(["dpkg", "-i", str(cloudflared_deb)], check=True)

tunnel_log_path = Path("/tmp/edgentrag-cloudflared.log")
tunnel_log = tunnel_log_path.open("w")
tunnel = subprocess.Popen(
    [
        "cloudflared", "tunnel", "--no-autoupdate",
        "--url", "http://127.0.0.1:8001",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

COLAB_URL = None
for _ in range(60):
    if tunnel.poll() is not None:
        break
    tunnel_log.flush()
    tunnel_output = tunnel_log_path.read_text()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_output)
    if match:
        COLAB_URL = match.group(0)
        break
    time.sleep(1)

if not COLAB_URL:
    tunnel_log.flush()
    print(tunnel_log_path.read_text())
    raise RuntimeError("Could not start the Cloudflare tunnel. Check the log above.")

print("Copy this temporary URL to your local machine:")
print(COLAB_URL)

## Test from this notebook

The first embedding request downloads/initializes the model and can take a while. A successful response includes one 384-number vector for this model.

In [ ]:
import json

request = urllib.request.Request(
    COLAB_URL + "/embed",
    data=json.dumps({"texts": ["A test document chunk sent from Colab."]}).encode(),
    headers={
        "Authorization": "Bearer " + os.environ["EDGENTRAG_EMBEDDING_API_TOKEN"],
        "Content-Type": "application/json",
    },
    method="POST",
)
with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read().decode())

print("model:", result["model"])
print("dimensions:", result["dimensions"])
print("vector count:", len(result["embeddings"]))
print("first 8 values:", result["embeddings"][0][:8])

## Test from your Mac

Leave the notebook runtime and tunnel running. In a local terminal, paste the URL printed above and enter the same token stored in Colab Secrets when prompted. This tests the network/API contract; the local ingestion worker is not wired to call the service yet.

~~~zsh
COLAB_URL="https://paste-the-trycloudflare-url-here"
curl "$COLAB_URL/health"
read -s "TOKEN?Shared embedding token: "
echo
curl -X POST "$COLAB_URL/embed" -H "Authorization: Bearer $TOKEN" -H "Content-Type: application/json" -d '{"texts":["A test document chunk sent from my Mac."]}'
unset TOKEN
~~~

Do not put the real token in this notebook, a committed .env, a URL, or a shared notebook output. The request sends text to the Colab service over HTTPS; only send data you are allowed to process there.

## Stop the processes

When finished, run this cell (or disconnect/delete the runtime). The temporary URL stops working when the tunnel exits.

In [ ]:
for process in (tunnel, server):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
print("Embedding API and tunnel stopped.")